[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevin-blasiak-curtin/ISYS2001-Archive/blob/main/Module%2008%20-%20API/1_financial_chatbot_gemini.ipynb)

# Building a Financial Advisor Chatbot

**Talking to an AI model through the Gemini API**

## What you'll build

A small chatbot we'll call the Financial Sage. It answers everyday money questions in plain language, and it can read a file of transactions and talk about your spending.

By the end you will have sent your first request to a real AI service and got a reply back in your own code.

## The one idea behind this week

An API is a request-and-response contract. You send a request, and something on the other side sends back a structured response you can use.

You will meet two APIs over the next two tasks:

- **Gemini** (this worksheet): you send it text, it sends text back.
- **A stock service** (the lab ticket): you send it a company code, it sends back numbers you can chart.

Same shape, different contents. Keep that picture in your head and the rest is detail.

## Setup

In [1]:
# The Google Gen AI library is how our Python code talks to Gemini.
!pip install google-genai

### Get your own API key

An API key is a private password that proves you are allowed to use the service. You get your own free one from Google AI Studio.

1. Go to https://aistudio.google.com/api-keys and sign in with the Google account you already use for Colab.
2. Create an API key and copy it.
3. In Colab, open the Secrets panel (the key icon in the left sidebar).
4. Add a new secret named exactly `GOOGLE_API_KEY`, paste your key into the value, and switch **Notebook access** on.

The name in the Secrets panel must match the name in the code, character for character. If it does not, you will get a "secret not found" error.

We store the key this way so it never sits inside a code cell where it could be shared or copied by accident.

In [3]:
from google.colab import userdata
from google import genai

# The client reads your key from Secrets, so the key itself never appears here.
client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

# The flash model is fast and free-tier friendly. The exact name changes every
# few months, so check AI Studio for the current one and edit this single line.
MODEL = "gemini-3.5-flash"

print("Connected to the Gemini API.")

Connected to the Gemini API.


### One small helper

Every time we want the model to answer something, we send a prompt and read the reply. Rather than write that out each time, we wrap it in a function once.

In [4]:
def ask_gemini(prompt):
    """Send a prompt to Gemini and return the text of its reply."""
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return response.text
    except Exception as error:
        return f"Something went wrong talking to the API: {error}"


# A quick test. If your key is set up, you should get a one-line answer back.
print(ask_gemini("In one sentence, what is compound interest?"))

Compound interest is the interest earned on both the initial principal amount and the accumulated interest from previous periods, essentially allowing your money to grow at an accelerating rate.


## Step 1: Think before you build

Before writing the chatbot, it helps to plan it. This is a good moment to use AI as a thinking partner. Try a prompt like this in AI Studio or Colab's assistant:

```
Help me plan a simple financial advisor chatbot that:
1. answers everyday money questions with general, educational information
2. can read transaction data from a CSV file
3. speaks in a friendly, plain tone
4. always reminds the user this is not professional advice

What functions would I need, and what should each one do?
```

Notice we are asking it to help us plan, not to write the whole thing. The plan is ours.

## Step 2: The Financial Sage

In [5]:
def financial_sage(question):
    """Answer a money question in the voice of a friendly, cautious guide."""
    personality = (
        "You are a friendly financial guide for university students. "
        "You give general, educational information about budgeting, saving "
        "and everyday money decisions. You keep answers short and plain. "
        "You always remind the reader that this is general information, "
        "not professional financial advice."
    )

    prompt = personality + "\n\nQuestion: " + question
    return ask_gemini(prompt)


# Try it
print(financial_sage("Should I keep my savings in a bank account or invest it?"))

Hey there! This is a great question that many students face. Here is a simple way to think about it:

*   **Keep it in a bank account (like a savings account) if:** You need the money soon (within the next 3 to 5 years) or for emergencies. It's safe, and you can get to it instantly if your laptop breaks or you need to pay rent.
*   **Invest it if:** You are thinking long-term (5+ years) and won't need this money anytime soon. Investing gives your money a chance to grow faster than inflation, but there is also a risk that you could lose money.

For most students, a good first step is building a small "emergency fund" in a bank account before thinking about investing. 

*Just a quick reminder: This is general educational information, not professional financial advice. If you need personalized help, it's always a good idea to speak with a qualified financial advisor!*


## Step 3: Some data to talk about

Let's make a small transactions file. Income is positive, spending is negative. We save it once so the rest of the notebook can read it.

In [6]:
import pandas as pd

transactions = {
    "Date": ["2024-01-01", "2024-01-05", "2024-01-06", "2024-01-07",
             "2024-01-10", "2024-01-12", "2024-01-15", "2024-01-18",
             "2024-01-20", "2024-01-22", "2024-01-25"],
    "Description": ["Salary", "Groceries Coles", "Electricity Bill", "Coffee Shop",
                    "Gym Membership", "Restaurant Dinner", "Rent Payment", "Movie Tickets",
                    "Groceries Coles", "Petrol", "Streaming Service"],
    "Category": ["Income", "Food", "Utilities", "Food",
                 "Health", "Dining", "Housing", "Entertainment",
                 "Food", "Transport", "Entertainment"],
    "Amount": [3000.00, -45.50, -120.00, -5.50,
               -50.00, -65.00, -1200.00, -30.00,
               -52.30, -60.00, -15.99],
}

df = pd.DataFrame(transactions)
df.to_csv("transactions.csv", index=False)
df

,Date,Description,Category,Amount
0,2024-01-01,Salary,Income,3000.00
1,2024-01-05,Groceries Coles,Food,-45.50
2,2024-01-06,Electricity Bill,Utilities,-120.00
3,2024-01-07,Coffee Shop,Food,-5.50
4,2024-01-10,Gym Membership,Health,-50.00
5,2024-01-12,Restaurant Dinner,Dining,-65.00
6,2024-01-15,Rent Payment,Housing,-1200.00
7,2024-01-18,Movie Tickets,Entertainment,-30.00
8,2024-01-20,Groceries Coles,Food,-52.30
9,2024-01-22,Petrol,Transport,-60.00


## Step 4: Turn the data into a summary

Before the model can say anything useful about your spending, we need to hand it a tidy summary rather than the raw rows. This function does the arithmetic with pandas and returns a short block of text.

In [7]:
def analyse_transactions(csv_file):
    """Read a transactions file and return a short written summary."""
    df = pd.read_csv(csv_file)

    total_income = df[df["Amount"] > 0]["Amount"].sum()
    total_expenses = abs(df[df["Amount"] < 0]["Amount"].sum())
    net_savings = total_income - total_expenses
    savings_rate = (net_savings / total_income * 100) if total_income > 0 else 0

    by_category = df[df["Amount"] < 0].groupby("Category")["Amount"].sum().abs()

    summary = f"""Financial summary:
- Total income: ${total_income:.2f}
- Total expenses: ${total_expenses:.2f}
- Net savings: ${net_savings:.2f}
- Savings rate: {savings_rate:.1f}%

Spending by category:
{by_category.to_string()}"""

    return summary


print(analyse_transactions("transactions.csv"))

Financial summary:
- Total income: $3000.00
- Total expenses: $1644.29
- Net savings: $1355.71
- Savings rate: 45.2%

Spending by category:
Category
Dining             65.00
Entertainment      45.99
Food              103.30
Health             50.00
Housing          1200.00
Transport          60.00
Utilities         120.00


## Step 5: Advice based on the data

Now we join the two ideas. We give the model the summary and a question at the same time, so its answer is grounded in the actual numbers.

In [8]:
def get_advice(csv_file, question):
    """Combine the transaction summary with a question and ask the sage."""
    summary = analyse_transactions(csv_file)

    prompt = f"""Here is the person's financial data:
{summary}

Their question: {question}

Give short, general, educational guidance based on this data."""

    return financial_sage(prompt)


print(get_advice("transactions.csv",
                  "Where is most of my money going, and what could I look at first?"))

Hey there! First off, high five! Saving over 45% of your income is absolutely amazing, especially as a student. You are in a great financial position. 

Here is a quick look at where your money is going:

*   **Where most of your money goes:** By far, your biggest expense is **Housing ($1,200)**, which eats up about 73% of your monthly spending. This is very common for students!
*   **What to look at first:** Since housing is usually a fixed cost (hard to change quickly), you might want to look at your next largest categories if you want to find savings:
    1.  **Utilities ($120):** You could look into simple energy-saving habits or shop around for cheaper internet/phone plans.
    2.  **Food & Dining (Combined $168.30):** Your food costs are actually quite low, but meal planning and buying in bulk are always great ways to keep these flexible costs down.

Keep up the fantastic work with your savings! 

*Just a quick reminder: This is general educational information to help you learn a

## Step 6: A simple chat loop

This ties it together into something you can talk to. Type `analyse` to see your summary, ask any money question, or type `quit` to stop.

In [19]:
def chat_with_sage():
    """A small interactive loop over the sage."""
    print("Financial Sage is ready.")
    print("Ask a money question, type 'analyse' to look at your transactions, or 'quit' to stop.")
    print()

    while True:
        user_input = input("You: ")

        if user_input.lower() == "quit":
            print("Sage: Take care with your money. Goodbye.")
            break
        elif user_input.lower() == "analyse":
            print("Sage: Here is your summary:")
            print(analyse_transactions("transactions.csv"))
        else:
            print("Sage:", get_advice("transactions.csv", user_input))

        print(analyse_transactions("transactions.csv"))

In [20]:
chat_with_sage()

Financial Sage is ready.
Ask a money question, type 'analyse' to look at your transactions, or 'quit' to stop.

You: analyse
Sage: Here is your summary:
Financial summary:
- Total income: $3000.00
- Total expenses: $1644.29
- Net savings: $1355.71
- Savings rate: 45.2%

Spending by category:
Category
Dining             65.00
Entertainment      45.99
Food              103.30
Health             50.00
Housing          1200.00
Transport          60.00
Utilities         120.00
Financial summary:
- Total income: $3000.00
- Total expenses: $1644.29
- Net savings: $1355.71
- Savings rate: 45.2%

Spending by category:
Category
Dining             65.00
Entertainment      45.99
Food              103.30
Health             50.00
Housing          1200.00
Transport          60.00
Utilities         120.00
You: quit
Sage: Take care with your money. Goodbye.


## Step 7: Try a few questions at once

A quick way to see how the sage handles different kinds of question. Questions that mention your spending get routed through the data; the rest get general answers.

In [22]:
questions = [
    "I have just started a part-time job. How should I think about saving?",
    "What is the difference between a debit card and a credit card?",
    "Based on my spending, is there anything I should watch?",
]

for q in questions:
    print("Question:", q)
    if "based on my spending" in q.lower():
        answer = get_advice("transactions.csv", q)
    else:
        answer = financial_sage(q)
    print("Sage:", answer[:300], "...")
    print("-" * 40)

Question: I have just started a part-time job. How should I think about saving?
Sage: Congrats on the new part-time job! That is a huge milestone. 

When it comes to saving, here is a simple way to think about it:

*   **Pay yourself first:** As soon as you get paid, move a small portion (like 10% or 20%) straight into a savings account. If you don't see it in your main account, you  ...
----------------------------------------
Question: What is the difference between a debit card and a credit card?
Sage: Hey there! This is one of the most common questions students ask, and it's super important to know the difference. 

Here is the quick breakdown:

*   **Debit Card (Your Money):** This card is linked directly to your bank account. When you buy something, the money is taken out of your account right  ...
----------------------------------------
Question: Based on my spending, is there anything I should watch?
Sage: Hey there! First off, congratulations—a **45.2% savings rate** is absol

## Your turn

Pick one small feature and build it, using AI as a thinking partner if you get stuck. You must be able to explain every line you keep.

Ideas:
- a savings goal tracker
- a "biggest three expenses" report
- a monthly budget check

Reflection prompt worth trying: paste your finished chatbot to an AI and ask, "What could go wrong with this, and what would you check before trusting its advice?"

In [31]:
# Your feature here.
pass
import datetime
import math
import sys
import time


def print_header(text):
    print("\n" + "=" * 55)
    print(f" {text}".center(55))
    print("=" * 55)


def get_float_input(prompt):
    while True:
        try:
            value = float(input(prompt).replace("$", "").replace(",", "").strip())
            if value < 0:
                print("⚠️  Please enter a positive amount.")
                continue
            return value
        except ValueError:
            print("⚠️  Invalid input. Please enter a valid number (e.g., 2500 or 150.50).")


def get_integer_input(prompt):
    while True:
        try:
            value = int(input(prompt).strip())
            if value <= 0:
                print("⚠️  Please enter a number greater than 0.")
                continue
            return value
        except ValueError:
            print("⚠️  Invalid input. Please enter a valid whole number.")


def display_progress_bar(current, target, length=30):
    percent = min(1.0, current / target) if target > 0 else 0
    filled_length = int(length * percent)
    bar = "█" * filled_length + "░" * (length - filled_length)
    percent_str = f"{percent * 100:.1f}%"
    print(f"\nProgress: [{bar}] {percent_str}")


def calculate_milestones(current, target):
    percentages = [0.25, 0.50, 0.75, 1.00]
    print("\n📌 Milestones Checklist:")
    for p in percentages:
        amount = target * p
        status = "✅ Reached!" if current >= amount else "⏳ In Progress"
        print(f"  • {int(p*100)}% (${amount:,.2f}): {status}")


def provide_coaching_message(percent):
    if percent >= 1.0:
        return "🎉 CONGRATULATIONS! You hit your goal! Time to celebrate your financial win!"
    elif percent >= 0.75:
        return "🔥 Home stretch! You're over 75% of the way there. Keep that momentum going!"
    elif percent >= 0.50:
        return "⭐ Halfway point crossed! You're making real, tangible progress."
    elif percent >= 0.25:
        return "🚀 Solid start! You've cleared the first 25%. Great habits build big results."
    else:
        return "🌱 Every journey starts with a single deposit. Consistency is key!"


def main():
    print_header("🎯 PERSONAL SAVINGS GOAL TRACKER")
    print("Welcome! Let's set up a plan to turn your financial targets into reality.\n")

    user_name = input("What's your name? ").strip()
    if not user_name:
        user_name = "Friend"

    goal_name = input(f"Hi {user_name}, what are you saving up for? (e.g., Emergency Fund, New Laptop, Vacation): ").strip()
    if not goal_name:
        goal_name = "Savings Goal"

    target_amount = get_float_input(f"What is the total cost for '{goal_name}'? $")
    current_savings = get_float_input("How much have you saved toward this already? $")

    print_header(f"📊 SAVINGS PROFILE: {goal_name.upper()}")

    remaining = max(0.0, target_amount - current_savings)
    percent_complete = (current_savings / target_amount) if target_amount > 0 else 0

    display_progress_bar(current_savings, target_amount)
    print(f"\nSaved: ${current_savings:,.2f} / ${target_amount:,.2f}")
    print(f"Remaining Target: ${remaining:,.2f}")

    print(f"\n💬 Coach Advice: {provide_coaching_message(percent_complete)}")
    calculate_milestones(current_savings, target_amount)

    if remaining > 0:
        print_header("🗓️ TIMELINE & CONTRIBUTION PLANNER")
        print("Let's calculate how quickly you can reach your goal!\n")
        print("1. Set a monthly deposit target")
        print("2. Set a timeframe target (in months)")
        choice = input("\nChoose an option (1 or 2): ").strip()

        if choice == "1":
            monthly_contrib = get_float_input("How much can you comfortably save each month? $")
            months_needed = math.ceil(remaining / monthly_contrib) if monthly_contrib > 0 else 0

            target_date = datetime.date.today() + datetime.timedelta(days=months_needed * 30.4)

            print(f"\n✨ Plan Summary:")
            print(f"• At ${monthly_contrib:,.2f}/month, you will reach your goal in roughly {months_needed} month(s).")
            print(f"• Estimated Completion Date: {target_date.strftime('%B %Y')}")

        elif choice == "2":
            target_months = get_integer_input("In how many months would you like to reach this goal? ")
            needed_per_month = remaining / target_months

            print(f"\n✨ Plan Summary:")
            print(f"• To hit your goal in {target_months} month(s), you need to save ${needed_per_month:,.2f} per month.")
            print(f"• That breaks down to about ${(needed_per_month / 4.33):,.2f} per week.")

    print_header("✨ KEEP UP THE GREAT WORK!")
    print(f"Good luck, {user_name}! Small, steady contributions add up fast.\n")


if __name__ == "__main__":
    main()


             🎯 PERSONAL SAVINGS GOAL TRACKER           
Welcome! Let's set up a plan to turn your financial targets into reality.

What's your name? darcy
Hi darcy, what are you saving up for? (e.g., Emergency Fund, New Laptop, Vacation): vacation
What is the total cost for 'vacation'? $5000
How much have you saved toward this already? $3000

               📊 SAVINGS PROFILE: VACATION             

Progress: [██████████████████░░░░░░░░░░░░] 60.0%

Saved: $3,000.00 / $5,000.00
Remaining Target: $2,000.00

💬 Coach Advice: ⭐ Halfway point crossed! You're making real, tangible progress.

📌 Milestones Checklist:
  • 25% ($1,250.00): ✅ Reached!
  • 50% ($2,500.00): ✅ Reached!
  • 75% ($3,750.00): ⏳ In Progress
  • 100% ($5,000.00): ⏳ In Progress

           🗓️ TIMELINE & CONTRIBUTION PLANNER          
Let's calculate how quickly you can reach your goal!

1. Set a monthly deposit target
2. Set a timeframe target (in months)

Choose an option (1 or 2): 1
How much can you comfortably save each

## Where we got to

You have:

- sent real requests to the Gemini API and read the replies
- kept your key out of your code using Colab Secrets
- combined pandas data work with an AI model to give grounded answers

The stock lab ticket uses the same request-and-response idea, only the response is numbers instead of text.

A reminder: this is educational content, not professional financial advice.